In [0]:
%pip install -q lightgbm==4.5.0 pyarrow
from pathlib import Path
from IPython.display import display
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import lightgbm as lgb

import mlflow
import mlflow.lightgbm

pd.set_option("display.float_format", "{:,.2f}".format)

### Data Loading

In [0]:
DATA = "/Volumes/workspace/forecasting/raw/"
SEED = 42           
HORIZON = 4         # forecast 4 weeks ahead
HOLDOUT_WEEKS = 12  # last 12 weeks = final holdout

train    = pd.read_csv(DATA + "train.csv", parse_dates=["Date"])
features = pd.read_csv(DATA + "features.csv", parse_dates=["Date"])
stores   = pd.read_csv(DATA + "stores.csv")

train.shape, features.shape, stores.shape

## Data preparation

In [0]:
weeks = np.sort(train["Date"].unique())
series = train[["Store", "Dept"]].drop_duplicates()

df = (
    series.merge(pd.DataFrame({"Date": weeks}), how="cross")
    .merge(train.drop(columns="IsHoliday"), on=["Store", "Dept", "Date"], how="left")
    .merge(stores, on="Store", how="left")
    .sort_values(["Store", "Dept", "Date"], ignore_index=True)
)
df["observed"] = df["Weekly_Sales"].notna()
df["IsHoliday"] = df["Date"].isin(train.loc[train["IsHoliday"], "Date"])

display(df.head())
pd.Series({"grid_rows": len(df), "observed_rows": df["observed"].sum(), "train_rows": len(train)},
          name="join_check")

In [0]:
bounds = df[df["observed"]].groupby(["Store", "Dept"])["Date"].agg(first="min", last="max")
span = df.join(bounds, on=["Store", "Dept"])
inside = span["Date"].between(span["first"], span["last"])

pd.Series({
    "duplicate_store_dept_weeks": train.duplicated(["Store", "Dept", "Date"]).sum(),
    "missing_values_train": train.isna().sum().sum(),
    "missing_values_stores": stores.isna().sum().sum(),
    "store_dept_series": len(bounds),
    "negative_sales_rows": (train["Weekly_Sales"] < 0).sum(),
    "zero_sales_rows": (train["Weekly_Sales"] == 0).sum(),
    "missing_weeks_inside_series": (inside & ~span["observed"]).sum(),
    "series_under_52_weeks": (df.groupby(["Store", "Dept"])["observed"].sum() < 52).sum(),
}, name="count")

In [0]:
# Gaps inside a series: are they missing records or weeks with (almost) no sales?
df["gap"] = inside & ~span["observed"]
same_series_prev = span["Store"].eq(span["Store"].shift()) & span["Dept"].eq(span["Dept"].shift())
same_series_next = span["Store"].eq(span["Store"].shift(-1)) & span["Dept"].eq(span["Dept"].shift(-1))
next_to_gap = span["observed"] & ((df["gap"].shift(1, fill_value=False) & same_series_prev)
                                  | (df["gap"].shift(-1, fill_value=False) & same_series_next))

pd.Series({
    "weeks next to a gap": next_to_gap.sum(),
    "median sales next to a gap ($)": df.loc[next_to_gap, "Weekly_Sales"].median(),
    "share under $100 (%)": 100 * (df.loc[next_to_gap, "Weekly_Sales"] < 100).mean(),
    "median sales, all weeks ($)": df["Weekly_Sales"].median(),
}).round(1)

- Duplicates and missing values: there are none in the raw data, so no cleaning step is needed.
- Negative and zero sales: I would keep returns. Because of these, the metric is WAPE (total error ÷ total sales) not MAPE, which divides by each week's sales.
- 27,667 missing weeks inside series: these are absent rows, not recorded values. Weeks next to a gap are tiny (median \$12, against \$7,612 for all weeks), so most gaps are probably weeks with little or no sales. They stay NaN and are skipped by rolling averages. Counting them as zero sales barely changes the results (checked in the error analysis).
- 340 series shorter than a year: these have no last-year value, so the seasonal naive falls back to the recent 4-week average.

### Time index and diagnostic plots

In [0]:
week_index = pd.DatetimeIndex(weeks)

pd.Series({
    "first_week": week_index.min().date(),
    "last_week": week_index.max().date(),
    "weeks": len(week_index),
    "days_between_weeks": sorted(set(np.diff(week_index).astype("timedelta64[D]").astype(int))),
    "weekday": sorted(set(week_index.day_name())),
}, name="time_index")

In [0]:
YEAR_COLORS = {2010: "#2a78d6", 2011: "#eb6834", 2012: "#1baf7a"}
plt.rcParams.update({"axes.spines.top": False, "axes.spines.right": False,
                     "axes.grid": True, "grid.color": "#e6e5e1"})

total = df.groupby("Date").agg(sales=("Weekly_Sales", "sum"), holiday=("IsHoliday", "any"))
total["sales"] /= 1e6
total["year"] = total.index.year
total["week"] = total.index.isocalendar().week.astype(int).to_numpy()

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(11, 7.5), constrained_layout=True)

ax1.plot(total.index, total["sales"], color=YEAR_COLORS[2010], lw=1.8)
holidays = total[total["holiday"]]
ax1.scatter(holidays.index, holidays["sales"], s=40, color="black", zorder=3, label="Flagged holiday week")
ax1.set(title="Total weekly sales, all 45 stores", ylabel="Sales ($M)")
ax1.legend(frameon=False, loc="upper left")

for year, d in total.groupby("year"):
    ax2.plot(d["week"], d["sales"], color=YEAR_COLORS[year], lw=1.8, label=year)
ax2.set(title="Same weeks, YoY", xlabel="week of the year", ylabel="Sales ($M)", xlim=(1, 53))
ax2.legend(frameon=False, title="Year", loc="upper left");

In [0]:
# Numbers behind the findings below
by_week = total.pivot_table(index="week", columns="year", values="sales")
nov_dec = total.index.month.isin([11, 12])


def same_weeks_change(sales_by_week, new, old):
    """% change in sales over the weeks present in both years."""
    pair = sales_by_week[[old, new]].dropna()
    return 100 * (pair[new].sum() / pair[old].sum() - 1)


display(total.nlargest(4, "sales")[["sales", "week", "holiday"]].round(1))
spring = total[total.index.month.isin([3, 4])]
display(spring.loc[spring.groupby("year")["sales"].idxmax(), ["sales", "week"]].round(1))

store_weeks = df.assign(week=df["Date"].dt.isocalendar().week, year=df["Date"].dt.year)
store_growth = (store_weeks.pivot_table(index=["Store", "week"], columns="year", values="Weekly_Sales", aggfunc="sum")
                .dropna(subset=[2010, 2012]).groupby("Store")[[2010, 2012]].sum())
store_change = 100 * (store_growth[2012] / store_growth[2010] - 1)

pd.Series({
    "median week ($M)": total["sales"].median(),
    "same weeks 2011 vs 2010 (%)": same_weeks_change(by_week, 2011, 2010),
    "same weeks 2012 vs 2011 (%)": same_weeks_change(by_week, 2012, 2011),
    "variation Nov-Dec (std / mean)": total.loc[nov_dec, "sales"].std() / total.loc[nov_dec, "sales"].mean(),
    "variation rest of year (std / mean)": total.loc[~nov_dec, "sales"].std() / total.loc[~nov_dec, "sales"].mean(),
    "store change 2012 vs 2010, lowest (%)": store_change.min(),
    "store change 2012 vs 2010, highest (%)": store_change.max(),
}).round(2)

In [0]:
stats = df[df["observed"]].groupby(["Store", "Dept"])["Weekly_Sales"].agg(["mean", "std", "count"])
stats = stats.query("count >= 52 and mean > 0 and std > 0")
slope = np.polyfit(np.log10(stats["mean"]), np.log10(stats["std"]), 1)[0]

fig, ax = plt.subplots(figsize=(6.5, 5), constrained_layout=True)
ax.scatter(stats["mean"], stats["std"], s=8, alpha=0.35, color=YEAR_COLORS[2010], edgecolor="none")
x = np.logspace(1, 5.5, 50)
ax.plot(x, x * (stats["std"] / stats["mean"]).median(), "k--", lw=1.5, label="std proportional to mean (slope 1)")
ax.set(xscale="log", yscale="log", title=f"Volatility grows with level (fitted slope {slope:.2f})",
       xlabel="Mean weekly sales per store-dept ($, log)", ylabel="Std of weekly sales ($, log)")
ax.legend(frameon=False, loc="upper left");

- Time index: weekly. Each date is the Friday ending the week: 143 weeks, exactly 7 days apart, with no missing dates in the total.
  
- Trend: flat in total at about \$46M a week. Same weeks changed −0.2% (2011 vs 2010) and +2.6% (2012 vs 2011). Stores differ much more, from −32% to +26% over two years, so the seasonal naive needs a growth adjustment per series (the momentum ratio in the baselines). No differencing or trend term needed.
- Seasonality: annual and very stable, since the three years overlap almost exactly. Two peaks dominate: Thanksgiving (week 47, about 66M) and the week before Christmas (week 51, 77–81M), against a median week of 46M. A smaller, unflagged bump moves between years: Easter (week 14 in 2012 vs weeks 13 and 16 earlier). The Christmas peak week also shifts by a day: it ends on Dec 24 in 2010 and on Dec 23 in 2011. This is why the model uses days-to-holiday features (including Easter) and the seasonal naive gets a Christmas calendar shift. Easter is not shifted in the seasonal naive, which is a limitation.
- Holiday flag: the data flags the week after Christmas (the sharp drop right after the peak), not the pre-Christmas peak. Results are reported for flagged holidays and for peak season (Thanksgiving through the week before Christmas).
- Variance: Nov–Dec is about 4× as volatile as the rest of the year (coefficient of variation 0.21 vs 0.05).
- Outliers: the spikes repeat on the same calendar events every year, so they aren't outliers and there are no sudden level shifts.
- Scale: across store-depts, volatility grows with sales level (log-log slope 0.75). This shows that series differ widely in scale, not that seasonality is multiplicative within a series. A log transform isn't possible because 1,285 weeks are negative and 73 are zero, so the model works on sales divided by each series' level, which puts all series on one scale.
- Slope below 1: small store-depts are relatively noisier than large ones. A single scaled interval won't fit all series sizes; this is checked when building the intervals.

### External variables

features.csv adds markdowns, temperature, fuel price, CPI and unemployment per store and week. None of them are used in the model. The two checks below show why.

In [0]:
# Weeks with markdown data available before each test period starts
markdown_weeks = features.loc[features["MarkDown1"].notna(), "Date"].unique()
test_starts = {"fold_1": "2011-08-12", "fold_2": "2011-11-11", "fold_3": "2012-02-10",
               "fold_4": "2012-05-11", "holdout": "2012-08-10"}
pd.Series({split: (markdown_weeks < pd.Timestamp(start)).sum() for split, start in test_starts.items()},
          name="weeks with markdown data before the period")

In [0]:
# Store weekly sales vs external variables, after removing store and week effects
store_week = (train.groupby(["Store", "Date"])["Weekly_Sales"].sum().reset_index()
              .merge(features, on=["Store", "Date"]))
store_week["log_sales"] = np.log(store_week["Weekly_Sales"])


def remove_store_and_week(frame, cols, n_iter=50):
    """Subtract store and week averages until both are zero (two-way fixed effects)."""
    out = frame[cols].astype(float)
    for _ in range(n_iter):
        out = out - out.groupby(frame["Store"]).transform("mean")
        out = out - out.groupby(frame["Date"]).transform("mean")
    return out


effects = {}
for var in ["Temperature", "CPI", "Unemployment"]:
    z = remove_store_and_week(store_week, ["log_sales", var])
    x, y = z[var].to_numpy(), z["log_sales"].to_numpy()
    slope = x @ y / (x @ x)
    per_store = pd.Series(x * (y - slope * x)).groupby(store_week["Store"].to_numpy()).sum()
    std_error = np.sqrt((per_store ** 2).sum()) / (x @ x)  # clustered by store
    effects[var] = {"sales change per unit (%)": 100 * slope, "t-value": slope / std_error}
pd.DataFrame(effects).T.round(2)

- Markdowns: recorded only from Nov 2011, so the first two test periods have no markdown history at all, and the others at most one holiday season. That is not enough to learn a reliable effect. Since promotions are planned ahead, they would be worth adding with more history.
- Temperature: warmer weeks sell slightly more (about +0.2% per °F, t = 5). But a forecast made 4 weeks ahead can't know that week's temperature, and the normal temperature for the time of year is already captured by week of year and last year's sales.
- CPI: no effect once store and week effects are removed.
- Unemployment: a weak effect, at the edge of chance (t = −2). It changes slowly and is published with a delay, so recent sales already reflect it.
- Fuel price: it moves the same way for every store, so its effect can't be separated from week effects here. Any national effect reaches the model through recent sales.

## Feature Engineering

**Data leakage prevention:** to forecast week *t*, every feature built from sales uses only weeks up to *t − 4*, the latest week known when the forecast is made.

- Lag features: sales 4, 5, 6, 8, 12 and 52 weeks ago.
- Rolling-window: average sales over the last 4, 8, 13 and 26 known weeks, and the standard deviation over the last 8. Missing weeks are skipped.
- Department-level features: average sales of the same department across all stores, 4 and 52 weeks ago.
- Calendar features: week of year, month, holiday flag, and days until Thanksgiving, Christmas and Easter. Moving holidays need their exact dates.
- Store features: store ID, department ID, store type and store size.
- Scaling: each series' level is the average of its last 13 known weeks. Series differ widely in scale, so the model predicts sales relative to this level, and the `_rel` features are key lags divided by the level.

In [0]:
KEYS = ["Store", "Dept"]
HOLIDAYS = {
    "thanksgiving": ["2010-11-25", "2011-11-24", "2012-11-22"],
    "christmas": ["2010-12-25", "2011-12-25", "2012-12-25"],
    "easter": ["2010-04-04", "2011-04-24", "2012-04-08"],
}


def days_to(dates, holiday_dates):
    """Days from each week's end date to that year's holiday (negative = holiday has passed)."""
    by_year = {d.year: d for d in pd.to_datetime(holiday_dates)}
    return (dates.dt.year.map(by_year) - dates).dt.days


def build_features(data):
    """Every sales-based feature uses only weeks at least HORIZON weeks before the target week."""
    out = data.copy()
    sales = out.groupby(KEYS)["Weekly_Sales"]
    for k in [4, 5, 6, 8, 12, 52]:
        out[f"lag_{k}"] = sales.shift(k)

    known = sales.shift(HORIZON).groupby([out["Store"], out["Dept"]])
    for w in [4, 8, 13, 26]:
        out[f"roll_mean_{w}"] = known.transform(lambda x: x.rolling(w, min_periods=1).mean())
    out["roll_std_8"] = known.transform(lambda x: x.rolling(8, min_periods=2).std())

    dept = out.groupby(["Dept", "Date"])["Weekly_Sales"].mean().groupby("Dept")
    dept_lags = pd.DataFrame({"dept_mean_lag_4": dept.shift(4), "dept_mean_lag_52": dept.shift(52)})
    out = out.join(dept_lags, on=["Dept", "Date"])

    out["week_of_year"] = out["Date"].dt.isocalendar().week.astype(int)
    out["month"] = out["Date"].dt.month
    for name, dates in HOLIDAYS.items():
        out[f"days_to_{name}"] = days_to(out["Date"], dates)

    out["level"] = out["roll_mean_13"].abs().fillna(out["lag_52"].abs()).fillna(10).clip(lower=10)
    for col in ["lag_4", "lag_52", "roll_mean_4", "dept_mean_lag_4", "dept_mean_lag_52"]:
        out[f"{col}_rel"] = out[col] / out["level"]
    return out


data = build_features(df)
data.shape

**Leakage test**

To forecast a week, the model uses sales from 4 or more weeks earlier. To test this, lets replace the sales of the target week and the 3 weeks before it with a fake value (1 billion) and rebuild the features.

- **Expected:** every feature for the target week stays the same, because none of them should use those weeks.
- **Control:** the target week's own sales (`Weekly_Sales`) must change. This proves the fake values went in.

This test covers the features. The training rows are checked separately after training.

In [0]:
cutoff = pd.Timestamp("2012-01-06")
tampered = df.copy()
tampered.loc[tampered["Date"] > cutoff - pd.Timedelta(weeks=HORIZON), "Weekly_Sales"] = 1e9

check_cols = ["Weekly_Sales"] + [c for c in data.columns if c not in df.columns]
before = data.loc[data["Date"] == cutoff, check_cols]
after = build_features(tampered).loc[lambda d: d["Date"] == cutoff, check_cols]

pd.Series((~np.isclose(before, after, equal_nan=True)).sum(axis=0), index=check_cols,
          name="rows_changed_by_tampering")

**Result:** only `Weekly_Sales` changed (in all 3,331 series). Every feature shows 0 changes, so no feature uses information that wouldn't be available 4 weeks ahead.

In [0]:
FEATURES = [
    "Store", "Dept", "Type", "Size", "IsHoliday", "week_of_year", "month",
    "days_to_thanksgiving", "days_to_christmas", "days_to_easter",
    "lag_4", "lag_5", "lag_6", "lag_8", "lag_12", "lag_52",
    "roll_mean_4", "roll_mean_8", "roll_mean_13", "roll_mean_26", "roll_std_8",
    "dept_mean_lag_4", "dept_mean_lag_52",
    "lag_4_rel", "lag_52_rel", "roll_mean_4_rel", "dept_mean_lag_4_rel", "dept_mean_lag_52_rel",
]
for col in ["Store", "Dept", "Type"]:
    data[col] = data[col].astype("category")

missing = data.loc[data["observed"], FEATURES].isna().mean()
(100 * missing[missing > 0]).round(1).sort_values(ascending=False).rename("% missing")

- 28 features: all 23 built from sales or dates pass the leakage test; the other 5 are store attributes and the holiday flag, which don't use sales.
- Missing values are structural:
  - The 52-week features are missing for about 36–38% of rows: the first year of data, plus series that start later.
  - Short lags are missing after calendar gaps and at the start of each series.
- No imputation: LightGBM handles missing values natively. Filling with 0 would incorrectly claim zero sales.
- Store, Dept and Type are categorical, so the model can learn per-department patterns without treating Dept 92 as "larger" than Dept 5.

## Baselines and Backtest Design

**Baselines.** Every model has to beat these forecasts, each built only from data known 4 weeks ahead:

- **Seasonal naive:** sales in the same week last year.
- **Moving average:** the average of the last 4 known weeks.
- **Adjusted seasonal naive (main benchmark):** last year's same week × growth. Growth = the last 13 known weeks ÷ the same 13 weeks a year earlier, capped between 0.5× and 2× so one odd period can't distort it.
- **Christmas shift:** each year, Christmas falls one or two weekdays later relative to the Friday week-ends (2 in leap years). So "the same week last year" covers slightly different shopping days around Christmas. For weeks ending Dec 8 – Jan 7, last year's value becomes a mix of that week and its neighbour, weighted by the real calendar difference (1 day for 2011 vs 2010; 2 days for 2012 vs 2011).

When last year's value doesn't exist (series younger than a year), every seasonal forecast falls back to the 4-week moving average.

In [0]:
def christmas_offset(dates):
    """Days from the Christmas of the same holiday season (January belongs to the previous Christmas)."""
    season_year = np.where(dates.dt.month >= 7, dates.dt.year, dates.dt.year - 1)
    return (dates - pd.to_datetime(pd.Series(season_year, index=dates.index).astype(str) + "-12-25")).dt.days


def add_baselines(frame):
    """Seasonal naive, 4-week moving average, adjusted seasonal naive, and the same with the Christmas shift."""
    out = frame.copy()
    sales = out.groupby(KEYS, observed=True)["Weekly_Sales"]
    level_last_year = out.groupby(KEYS, observed=True)["roll_mean_13"].shift(52)
    growth = (out["roll_mean_13"] / level_last_year.where(level_last_year > 0)).clip(0.5, 2).fillna(1)

    offset = christmas_offset(out["Date"])
    out["shift_days"] = (christmas_offset(out["Date"] - pd.Timedelta(weeks=52)) - offset).where(offset.between(-17, 13), 0)
    neighbour = np.where(out["shift_days"] > 0, sales.shift(53), sales.shift(51))
    weight = out["shift_days"].abs() / 7
    out["last_year_shifted"] = ((1 - weight) * out["lag_52"] + weight * neighbour).fillna(out["lag_52"])

    out["ma4"] = out["roll_mean_4"].fillna(0)
    out["snaive"] = out["lag_52"].fillna(out["ma4"])
    out["asn"] = (out["lag_52"] * growth).fillna(out["ma4"])
    out["asn_xmas"] = (out["last_year_shifted"] * growth).fillna(out["ma4"])
    return out


data = add_baselines(data)

check = (data.loc[data["Date"].between("2011-12-09", "2012-01-06") & data["observed"]]
         .groupby("Date")
         .agg(shift_days=("shift_days", "first"), actual=("Weekly_Sales", "sum"),
              last_year=("lag_52", "sum"), last_year_shifted=("last_year_shifted", "sum")))
check[["actual", "last_year", "last_year_shifted"]] /= 1e6
check.round(1)

In [0]:
N_FOLDS, FOLD_WEEKS = 4, 13
HOLIDAY_NAMES = {2: "Super Bowl", 9: "Labor Day", 11: "Thanksgiving", 12: "Christmas"}
first_test = len(weeks) - HOLDOUT_WEEKS - N_FOLDS * FOLD_WEEKS

split_of_week = pd.Series("train_only", index=weeks)
for i in range(N_FOLDS):
    split_of_week.iloc[first_test + i * FOLD_WEEKS: first_test + (i + 1) * FOLD_WEEKS] = f"fold_{i + 1}"
split_of_week.iloc[-HOLDOUT_WEEKS:] = "holdout"

data["split"] = data["Date"].map(split_of_week)
data["peak"] = (data["days_to_thanksgiving"] <= 0) & (data["days_to_christmas"] >= 0)

holidays = (data.loc[data["IsHoliday"], ["split", "Date"]].drop_duplicates()
            .groupby("split")["Date"]
            .agg(holiday_weeks="count", holidays=lambda x: ", ".join(x.dt.month.map(HOLIDAY_NAMES).unique())))
overview = data.groupby("split", sort=False)["Date"].agg(first_week="min", last_week="max", weeks="nunique")
overview.join(holidays).fillna({"holiday_weeks": 0, "holidays": "none"}).astype({"holiday_weeks": int})

In [0]:
def wape(frame, col):
    """Total absolute error divided by total sales."""
    return (frame["Weekly_Sales"] - frame[col]).abs().sum() / frame["Weekly_Sales"].abs().sum()


def season_of(frame):
    """Season of each week (the holdout is kept separate)."""
    season = pd.cut(frame["Date"].dt.month, [0, 4, 7, 10, 12], labels=["Jan-Apr", "May-Jul", "Aug-Oct", "Nov-Dec"])
    return season.astype(str).where(frame["split"] != "holdout", "Aug-Oct (holdout)")


def score(frame, models):
    """WAPE (%) per model on the backtest folds and their holiday and peak weeks. The holdout is scored once, at the end."""
    obs = frame[frame["observed"] & frame["split"].str.startswith("fold")]
    subsets = {"backtest": obs, "backtest_holiday_flag": obs[obs["IsHoliday"]], "backtest_peak_season": obs[obs["peak"]]}
    return pd.DataFrame({name: [wape(sub, col) for col in models.values()] for name, sub in subsets.items()},
                        index=list(models)).mul(100).round(1)


BASELINES = {"Seasonal naive": "snaive", "Moving average (4 wk)": "ma4",
             "Adjusted seasonal naive": "asn", "Adjusted seasonal naive + Christmas shift": "asn_xmas"}
score(data, BASELINES)

**Backtest design**
- 4 test periods of 13 weeks (Aug 2011 – Aug 2012) cover a full year: Labor Day, Thanksgiving + Christmas, Super Bowl, and summer.
- Before each period the model is retrained on the weeks known at the period's first forecast, so it stops 4 weeks before the period starts.
- Each week is forecast 4 weeks ahead: inside a test period, a forecast may use actual sales up to 4 weeks earlier, as a planner would.
- The last 12 weeks (Aug–Oct 2012) are the final holdout. It is scored once, at the end, after every decision is fixed.
- Holiday evidence is thin: the backtest contains only one Thanksgiving and one Christmas, so holiday results are not used to choose between models.

**Baseline results**
- The moving average is the weakest (18.1%) and collapses in peak season (27.3%). It has no memory of last year's holidays.
- The adjusted seasonal naive (10.4%) beats plain seasonal naive (11.6%) because it scales last year by recent growth. It is the main benchmark.
- The Christmas shift corrects much of the holiday misalignment. For the week ending Dec 30, 2011, last year's value was 40.4M; shifted, it becomes 46.2M against 46.0M actual. Backtest error drops to 10.2%, and peak season to 11.4%.

## Model Training and Evaluation

**LightGBM** One model for all 3,331 store-depts, retrained before each test period.

- What it predicts: sales ÷ the series' level, not raw dollars, so all series are on one scale. Weighting each row by its level keeps the loss in dollars, so large departments still count more.
- Target cap: the ratio is capped at −1 to 20 and the level has a floor of 10, so tiny or brand-new series can't produce extreme targets. This changes about 3% of training rows, almost all in a series' first 4 weeks (checked below).
- Loss: the model predicts the median (quantile loss with α = 0.5). WAPE adds up absolute errors, and the median is the forecast that makes total absolute error smallest.
- Settings: fixed at 600 trees, learning rate 0.05 and 127 leaves, not tuned. Two other settings are checked below. The `deterministic` setting makes reruns on the same machine give identical numbers.

**Final forecast** A 50-50 blend of LightGBM and the adjusted seasonal naive with the Christmas shift. This is a pre-set hedge: it pairs the model with a simple forecast that carries last year's holiday shape. The weight is fixed at 0.5, not tuned.

**Selection** Use the Christmas shift only if it lowers overall backtest WAPE. Holiday weeks aren't used to choose, because the backtest contains only one Thanksgiving and one Christmas.

In [0]:
PARAMS = {
    "objective": "quantile", "alpha": 0.5,
    "learning_rate": 0.05, "num_leaves": 127, "min_data_in_leaf": 50,
    "feature_fraction": 0.8, "bagging_fraction": 0.8, "bagging_freq": 1,
    "seed": SEED, "deterministic": True, "force_row_wise": True, "verbose": -1,
}
N_ROUNDS = 600
TEST_SPLITS = [f"fold_{i + 1}" for i in range(N_FOLDS)] + ["holdout"]
FOLDS = TEST_SPLITS[:-1]


def fit_lgbm(rows, params=PARAMS, rounds=N_ROUNDS):
    """Learn sales relative to each series' level; weighting by level keeps the loss in dollars."""
    target = (rows["Weekly_Sales"] / rows["level"]).clip(-1, 20)
    weight = rows["level"] / rows["level"].mean()
    return lgb.train(params, lgb.Dataset(rows[FEATURES], target, weight=weight), rounds)


def training_rows(split):
    """Observed weeks known at the split's first forecast, i.e. HORIZON weeks before its first week."""
    first_week = data.loc[data["split"] == split, "Date"].min()
    return observed[observed["Date"] <= first_week - pd.Timedelta(weeks=HORIZON)]


observed = data[data["observed"]]
models = {}
for split in TEST_SPLITS:
    test = data["split"] == split
    models[split] = fit_lgbm(training_rows(split))
    data.loc[test, "lgbm"] = models[split].predict(data.loc[test, FEATURES]) * data.loc[test, "level"]

data["blend_plain"] = 0.5 * data["lgbm"] + 0.5 * data["asn"]
data["blend"] = 0.5 * data["lgbm"] + 0.5 * data["asn_xmas"]

tested = data[data["observed"] & (data["split"] != "train_only")]
backtest = tested[tested["split"].isin(FOLDS)]
(backtest.groupby("split")
 .apply(lambda s: pd.Series({m: wape(s, m) for m in ["asn_xmas", "lgbm", "blend"]}), include_groups=False)
 .mul(100).round(1))

In [0]:
# Leakage check for training rows: every model must stop at least HORIZON weeks before its first forecast week
training_check = pd.DataFrame({
    "last_training_week": {s: training_rows(s)["Date"].max() for s in TEST_SPLITS},
    "first_forecast_week": {s: data.loc[data["split"] == s, "Date"].min() for s in TEST_SPLITS},
})
training_check["weeks_between"] = (training_check["first_forecast_week"] - training_check["last_training_week"]).dt.days // 7
display(training_check)

# How many training rows the target cap (-1 to 20) changes, and where they come from
ratio = observed["Weekly_Sales"] / observed["level"]
capped = (ratio < -1) | (ratio > 20)
week_in_series = observed.groupby(KEYS, observed=True).cumcount()
pd.Series({
    "training rows capped (%)": 100 * capped.mean(),
    "of which in a series' first 4 weeks (%)": 100 * (week_in_series[capped] < 4).mean(),
}).round(1)

In [0]:
# Sensitivity: two other settings, backtest only (reported, not used to choose)
ALTERNATIVES = {
    "LightGBM defaults (31 leaves, 100 trees, lr 0.1)": (
        {**PARAMS, "num_leaves": 31, "learning_rate": 0.1, "min_data_in_leaf": 20,
         "feature_fraction": 1.0, "bagging_fraction": 1.0, "bagging_freq": 0}, 100),
    "Smaller trees (31 leaves, 600 trees)": ({**PARAMS, "num_leaves": 31}, N_ROUNDS),
}
sensitivity = {"Chosen (127 leaves, 600 trees)": {"lightgbm": wape(backtest, "lgbm"), "blend": wape(backtest, "blend")}}
for name, (params, rounds) in ALTERNATIVES.items():
    alt = pd.Series(np.nan, index=backtest.index)
    for split in FOLDS:
        rows = backtest["split"] == split
        model = fit_lgbm(training_rows(split), params, rounds)
        alt[rows] = model.predict(backtest.loc[rows, FEATURES]) * backtest.loc[rows, "level"]
    alt_frame = backtest.assign(alt=alt, alt_blend=0.5 * alt + 0.5 * backtest["asn_xmas"])
    sensitivity[name] = {"lightgbm": wape(alt_frame, "alt"), "blend": wape(alt_frame, "alt_blend")}
pd.DataFrame(sensitivity).T.mul(100).round(2).add_suffix(" backtest WAPE %")

In [0]:
christmas_rule = pd.Series({
    "blend, no Christmas shift": wape(backtest, "blend_plain"),
    "blend, Christmas shift": wape(backtest, "blend"),
}, name="backtest WAPE %").mul(100).round(2)
display(christmas_rule)

MODELS = {**BASELINES, "LightGBM": "lgbm", "Blend, no Christmas shift": "blend_plain", "Blend 50-50": "blend"}
score(data, MODELS)

- Training rows: every model stops 4 weeks before its first forecast week, so no model has seen sales a planner wouldn't have.
- Settings: both alternatives give a similar or slightly worse backtest, so the results don't depend on the chosen settings.
- Christmas shift: it lowers backtest WAPE (9.12% to 9.03%), so it is kept. From here on, the benchmark is the adjusted seasonal naive with the Christmas shift, so the shift's gain is not credited to the model.

In [0]:
by_season = backtest.groupby(season_of(backtest)).apply(
    lambda s: pd.Series({"benchmark": wape(s, "asn"), "benchmark_with_shift": wape(s, "asn_xmas"), "blend": wape(s, "blend")}))
by_season["error_cut_vs_shifted"] = 1 - by_season["blend"] / by_season["benchmark_with_shift"]
by_season.mul(100).round(1)

In [0]:
def weekly_errors(frame, base, model):
    """Weekly totals of absolute error for a base forecast and a model, plus sales and key-week flag."""
    return (frame.assign(err_base=(frame["Weekly_Sales"] - frame[base]).abs(),
                         err_model=(frame["Weekly_Sales"] - frame[model]).abs(),
                         sales=frame["Weekly_Sales"].abs(),
                         key=frame["IsHoliday"] | frame["peak"])
            .groupby("Date")[["err_base", "err_model", "sales", "key"]].sum())


def block_bootstrap(weekly, n_boot=5000, block=4):
    """Gain of the model over the base in WAPE points, with a 95% interval from resampling 4-week blocks."""
    rng = np.random.default_rng(SEED)
    n_weeks = len(weekly)
    starts = rng.integers(0, n_weeks - block + 1, size=(n_boot, -(-n_weeks // block)))
    idx = (starts[:, :, None] + np.arange(block)).reshape(n_boot, -1)[:, :n_weeks]
    sums = {c: weekly[c].to_numpy()[idx].sum(axis=1) for c in ["err_base", "err_model", "sales"]}
    gain = 100 * (sums["err_base"] - sums["err_model"]) / sums["sales"]
    key = weekly[weekly["key"] > 0]
    return pd.Series({
        "gain_points": 100 * (weekly["err_base"].sum() - weekly["err_model"].sum()) / weekly["sales"].sum(),
        "ci_95_low": np.percentile(gain, 2.5),
        "ci_95_high": np.percentile(gain, 97.5),
        "weeks_won": (weekly["err_model"] < weekly["err_base"]).sum(),
        "weeks": n_weeks,
        "key_weeks_gain_points": 100 * (key["err_base"].sum() - key["err_model"].sum()) / key["sales"].sum(),
        "key_weeks_won": (key["err_model"] < key["err_base"]).sum(),
        "key_weeks": len(key),
    })


# blend vs the benchmark with the Christmas shift
weekly = weekly_errors(backtest, "asn_xmas", "blend")
block_bootstrap(weekly).round(2).rename("blend vs benchmark with shift")

In [0]:
# Blend vs LightGBM alone, and how the backtest changes with the blend weight (reported, not used to choose)
display(block_bootstrap(weekly_errors(backtest, "lgbm", "blend")).round(2).rename("blend vs LightGBM alone"))

pd.Series({w: wape(backtest.assign(mix=w * backtest["lgbm"] + (1 - w) * backtest["asn_xmas"]), "mix")
           for w in [0, 0.25, 0.5, 0.75, 1]}, name="backtest WAPE %").mul(100).round(2).rename_axis("LightGBM weight")

In [0]:
# WAPE per key week (holiday flag or peak season)
key = weekly[weekly["key"] > 0]
pd.DataFrame({"benchmark_with_shift": 100 * key["err_base"] / key["sales"],
              "blend": 100 * key["err_model"] / key["sales"]}).round(1)

**Results**
- Overall: the blend lowers backtest WAPE from 10.2% (benchmark with the Christmas shift) to 9.0%. The 95% bootstrap interval for the gain is 0.9–1.5 points. It wins 50 of 52 weeks and all 4 test periods.
- By season: error drops 9–15% in every season, least in Nov–Dec and Aug–Oct.
- Holidays: LightGBM alone is worse than the benchmark on holidays (12.8% vs 10.2%), likely because it has seen only one earlier Thanksgiving and Christmas. The blend is about level with the benchmark on flagged holidays (10.0% vs 10.2%) and better in peak season (10.4% vs 11.4%). It wins 7 of 8 key weeks and loses Thanksgiving 2011.
- Blend vs LightGBM alone: the blend is slightly better overall, but the difference is within chance (interval −0.2 to +0.8 points). Its advantage is in the holiday weeks, where it wins all 8. A higher LightGBM weight would do about as well, but the weight stays at the pre-set 0.5.
- Caveat: the backtest has only one Thanksgiving and one Christmas, so holiday results are indicative.

## Error Analysis

Errors on the 4 backtest periods, for the blend and the benchmark with the Christmas shift. The holdout is left for the end.

In [0]:
errors = backtest.copy()
errors["err_blend"] = (errors["Weekly_Sales"] - errors["blend"]).abs()
errors["err_bench"] = (errors["Weekly_Sales"] - errors["asn_xmas"]).abs()
errors["abs_sales"] = errors["Weekly_Sales"].abs()


def error_breakdown(by):
    """Share of total error vs share of sales, and WAPE of blend and benchmark, per group (%)."""
    totals = errors.groupby(by, observed=True)[["err_blend", "err_bench", "abs_sales"]].sum()
    return pd.DataFrame({
        "error_share": totals["err_blend"] / totals["err_blend"].sum(),
        "sales_share": totals["abs_sales"] / totals["abs_sales"].sum(),
        "wape_blend": totals["err_blend"] / totals["abs_sales"],
        "wape_benchmark": totals["err_bench"] / totals["abs_sales"],
    }).mul(100).round(1)


by_dept = error_breakdown("Dept").sort_values("error_share", ascending=False)
by_dept.head(10)

In [0]:
store_info = data.groupby("Store", observed=True)[["Type", "Size"]].first()
display(error_breakdown("Type"))

by_store = error_breakdown("Store").join(store_info).sort_values("wape_blend", ascending=False)
pd.concat({"worst 5": by_store.head(5), "best 5": by_store.tail(5)})

In [0]:
weekly_wape = errors.groupby("Date")[["err_blend", "err_bench", "abs_sales"]].sum()
weekly_wape = weekly_wape[["err_blend", "err_bench"]].div(weekly_wape["abs_sales"], axis=0) * 100
key_dates = errors.loc[errors["IsHoliday"] | errors["peak"], "Date"].unique()

fig, ax = plt.subplots(figsize=(11, 4.5), constrained_layout=True)
for d in key_dates:
    ax.axvspan(d - pd.Timedelta(days=3), d + pd.Timedelta(days=3), color="#e6e5e1", zorder=0)
ax.plot(weekly_wape.index, weekly_wape["err_bench"], color=YEAR_COLORS[2011], lw=1.8,
        label="Benchmark (adjusted seasonal naive + Christmas shift)")
ax.plot(weekly_wape.index, weekly_wape["err_blend"], color=YEAR_COLORS[2010], lw=1.8, label="Blend (final model)")
ax.annotate("Easter 2012\n(Easter 2011 fell\n2 weeks later)", (pd.Timestamp("2012-04-20"), weekly_wape.loc["2012-04-20", "err_blend"]),
            xytext=(pd.Timestamp("2012-05-20"), 18), fontsize=9, arrowprops={"arrowstyle": "->", "lw": 0.8})
ax.set(title="Weekly WAPE across the backtest (grey bands = holiday and peak weeks)", ylabel="WAPE (%)", ylim=(0, 21))
ax.legend(frameon=False, loc="upper left");

In [0]:
dept_weekly = data[data["observed"]].groupby(["Dept", "Date"], observed=True)["Weekly_Sales"].sum()
by_dept["seasonality"] = dept_weekly.groupby("Dept", observed=True).agg(lambda s: s.std() / s.mean())
main = by_dept[by_dept["sales_share"] >= 0.2]  # departments with at least 0.2% of sales

fig, ax = plt.subplots(figsize=(7.5, 5), constrained_layout=True)
ax.scatter(main["seasonality"], main["wape_blend"], s=main["sales_share"] * 40, alpha=0.6,
           color=YEAR_COLORS[2010], edgecolor="white", lw=0.5)
for dept, row in main.head(5).iterrows():  # the 5 departments with the largest share of total error
    ax.annotate(f"Dept {dept}", (row["seasonality"], row["wape_blend"]),
                xytext=(5, 4), textcoords="offset points", fontsize=9)
corr = main[["seasonality", "wape_blend"]].corr().iloc[0, 1]
ax.set(title=f"Seasonal departments are harder to forecast (correlation {corr:.2f})",
       xlabel="Department seasonality (std ÷ mean of weekly sales)", ylabel="Blend WAPE (%)");

In [0]:
seasonal_depts = by_dept.index[by_dept["seasonality"] > 0.4]
observed_rows = data[data["observed"]]
mix = (observed_rows
       .assign(seasonal_sales=observed_rows["Weekly_Sales"].where(observed_rows["Dept"].isin(seasonal_depts), 0))
       .groupby("Type", observed=True)[["seasonal_sales", "Weekly_Sales"]].sum())
dept_type = errors.groupby(["Dept", "Type"], observed=True)[["err_blend", "abs_sales"]].sum()
dept_type_wape = (dept_type["err_blend"] / dept_type["abs_sales"]).unstack()

pd.Series({
    "top 10 departments: share of total error (%)": by_dept["error_share"].head(10).sum(),
    "highly seasonal departments (seasonality > 0.4)": len(seasonal_depts),
    **{f"type {t}: % of sales from those departments": 100 * r["seasonal_sales"] / r["Weekly_Sales"]
       for t, r in mix.iterrows()},
    "type B vs A, same department: median WAPE ratio": (dept_type_wape["B"] / dept_type_wape["A"]).median(),
}).round(2)

In [0]:
# Gap weeks: results if the missing weeks inside the test periods were zero sales
gap_rows = data[data["gap"] & data["split"].isin(FOLDS)].assign(Weekly_Sales=0.0)
with_gaps = pd.concat([backtest, gap_rows])
pd.DataFrame({
    "gaps skipped": [wape(backtest, m) for m in ["asn_xmas", "blend"]],
    "gaps as zero sales": [wape(with_gaps, m) for m in ["asn_xmas", "blend"]],
}, index=["benchmark with shift", "blend"]).mul(100).round(2)

**Where the model fails and why**

- Departments: a few departments cause a large share of all error (40% from the top 10). Dept 72 is the biggest single source. Large, steady departments (92, 95, 38) add a lot of error in dollars only because they are big; their error rate is low.
- Why: error rises with how seasonal a department is (seasonality chart). Highly seasonal departments depend on a few peak weeks, where small changes in timing or size cause large errors. Department names are anonymized, so the products are unknown.
- Stores: type B stores are the hardest, and all 5 worst stores are type B. This is mostly their department mix: they sell more from highly seasonal departments. Within the same department, B stores are only slightly worse than A stores.
- Weeks: the worst weeks are around Easter 2012 (weekly chart). Easter came 16 days earlier than in 2011, so "the same week last year" puts the Easter bump in the wrong week. LightGBM knows the days to Easter; the seasonal naive does not, which is a limitation.
- Gap weeks: counting the missing weeks as zero sales changes WAPE by less than 0.05 points.

### Forecast bias

Before building ranges and stock levels, to check whether the forecasts are systematically too high or too low.

In [0]:
def bias(frame):
    """(forecast - actual) / actual, in % of sales. Negative = under-forecast."""
    actual = frame["Weekly_Sales"].sum()
    return pd.Series({m: 100 * (frame[m].sum() - actual) / actual for m in ["asn_xmas", "lgbm", "blend"]})


size_group = pd.qcut(backtest["level"], 5, labels=["smallest", "small", "middle", "large", "largest"])
pd.concat({
    "test period": backtest.groupby("split").apply(bias, include_groups=False),
    "season": backtest.groupby(season_of(backtest)).apply(bias),
    "series size": backtest.groupby(size_group, observed=True).apply(bias),
}).round(2)

In [0]:
# Bias correction test: scale each forecast by its size group's bias, learned on the other test periods
corrected = backtest.assign(size_group=size_group)
for col in ["lgbm", "blend"]:
    for split in FOLDS:
        other = corrected[corrected["split"] != split]
        totals = other.groupby("size_group", observed=True)[["Weekly_Sales", col]].sum()
        factor = totals["Weekly_Sales"] / totals[col]
        rows = corrected["split"] == split
        corrected.loc[rows, f"{col}_corrected"] = corrected.loc[rows, col] * corrected.loc[rows, "size_group"].map(factor).astype(float)


def bias_pct(frame, col):
    return 100 * (frame[col].sum() - frame["Weekly_Sales"].sum()) / frame["Weekly_Sales"].sum()


by_period = {c: corrected.groupby("split").apply(lambda s: bias_pct(s, c)) for c in ["lgbm", "lgbm_corrected", "blend", "blend_corrected"]}
pd.DataFrame({
    "backtest WAPE %": {c: 100 * wape(corrected, c) for c in by_period},
    "bias %": {c: bias_pct(corrected, c) for c in by_period},
    "bias by period, lowest %": {c: v.min() for c, v in by_period.items()},
    "bias by period, highest %": {c: v.max() for c, v in by_period.items()},
}).round(2)

- LightGBM under-forecasts in every test period (by 1–3.5% of sales), and most for small series. It predicts the median, and for skewed sales the median sits below the average.
- The blend roughly halves this, since the seasonal naive tends to over-forecast slightly. It still under-forecasts in fold 1 (−3.1%) and for small series.
- A correction that scales each size group by its past bias removes the overall bias but makes WAPE worse, and the bias still swings between periods. So it is not applied: WAPE rewards the median, and the ranges and stock levels below are calibrated on actual errors, so the under-forecast is built into their buffers.

## Uncertainty Quantification

**Forecast ranges (P10–P90)** Each series gets an error scale: the adjusted seasonal naive's average absolute error over weeks *t − 16* to *t − 4*, all known when forecasting week *t*. The range is the blend forecast plus a low and a high multiple of that scale. The multiples are the 10th and 90th percentiles of the blend's past errors divided by their scale, taken from the other backtest periods (all four for the holdout).

**Rule** The 80% range passes if actual sales fall inside it 75–85% of the time in each of 10 size groups, and 75–85% of sales value falls inside it. Coverage by season and in holiday weeks is reported but not used to decide. If the rule fails, no stocking recommendation is made. Results here are for the backtest; the holdout is checked at the end.

In [0]:
def add_error_scale(frame):
    """The adjusted naive's mean absolute error over weeks t-16..t-4, all known when forecasting week t."""
    out = frame.copy()
    naive_abs_err = (out["Weekly_Sales"] - out["asn_xmas"]).abs()
    out["err_scale"] = (naive_abs_err.groupby([out["Store"], out["Dept"]], observed=True)
                        .transform(lambda s: s.shift(HORIZON).rolling(13, min_periods=1).mean())
                        .fillna(0.1 * out["level"])  # no recent errors: 10% of the series level
                        .clip(lower=1))              # no zero-width ranges for tiny series
    return out


data = add_error_scale(data)

tested = data[data["observed"] & (data["split"] != "train_only")].copy()
tested["ratio"] = (tested["Weekly_Sales"] - tested["blend"]) / tested["err_scale"]

# each test period is calibrated on the other backtest folds (the holdout on all four)
for split in TEST_SPLITS:
    calibration = tested[tested["split"].isin(FOLDS) & (tested["split"] != split)]
    low, high = calibration["ratio"].quantile([0.1, 0.9])
    rows = tested["split"] == split
    tested.loc[rows, "p10"] = tested.loc[rows, "blend"] + low * tested.loc[rows, "err_scale"]
    tested.loc[rows, "p90"] = tested.loc[rows, "blend"] + high * tested.loc[rows, "err_scale"]

tested["inside"] = tested["Weekly_Sales"].between(tested["p10"], tested["p90"])
tested["size_group"] = pd.qcut(tested["level"], 10, labels=range(1, 11))  # 1 = smallest series
tested["period"] = np.where(tested["split"] == "holdout", "holdout", "backtest")

(tested[tested["period"] == "backtest"].groupby("size_group", observed=True)["inside"].mean()
 .mul(100).round(1).rename("% of weeks inside P10-P90"))

In [0]:
def interval_summary(frame):
    """Rule checks, plus the interval score (rewards narrow ranges, penalises misses)."""
    actual, sales = frame["Weekly_Sales"], frame["Weekly_Sales"].abs()
    by_size = frame.groupby("size_group", observed=True)["inside"].mean()
    miss = (frame["p10"] - actual).clip(lower=0) + (actual - frame["p90"]).clip(lower=0)
    return pd.Series({
        "lowest_size_group_%": 100 * by_size.min(),
        "highest_size_group_%": 100 * by_size.max(),
        "sales_weighted_%": 100 * (frame["inside"] * sales).sum() / sales.sum(),
        "below_p90_%": 100 * (actual <= frame["p90"]).mean(),
        "interval_score": ((frame["p90"] - frame["p10"]).sum() + 10 * miss.sum()) / sales.sum(),
    })


summary = interval_summary(tested[tested["period"] == "backtest"])
display(summary.round(2))
pd.Series({"passes_rule": summary["lowest_size_group_%"] >= 75 and summary["highest_size_group_%"] <= 85
           and 75 <= summary["sales_weighted_%"] <= 85})

In [0]:
def coverage(frame):
    sales = frame["Weekly_Sales"].abs()
    return pd.Series({"rows_%": 100 * frame["inside"].mean(),
                      "sales_weighted_%": 100 * (frame["inside"] * sales).sum() / sales.sum()})


backtest_ranges = tested[tested["period"] == "backtest"]
report = backtest_ranges.groupby(season_of(backtest_ranges)).apply(coverage)
report.loc["Key weeks"] = coverage(backtest_ranges[backtest_ranges["IsHoliday"] | backtest_ranges["peak"]])
report.round(1)

The ranges pass the rule on the backtest: coverage stays within 75–85% in every size group and by sales value. Actual sales stay below P90 about 90% of the time, as intended. Scaling by each series' own recent error keeps coverage even across sizes.

Limitation: in Nov–Dec the ranges are too narrow and cover only about 71% of weeks. One set of multiples for the whole year can't capture the extra volatility of peak weeks. A separate Nov–Dec calibration couldn't be checked, because 8 of the 9 Nov–Dec backtest weeks are in one test period.

**Follow-up: wider ranges at peak** The error scale comes from the 13 weeks before *t − 4*, which are calm weeks just before a peak, so peak ranges come out too narrow. Fix tested: multiply the scale by forecast ÷ level (limited to 0.5–3), so the range widens when the forecast expects a spike.

**Rule** Adopt only if the new ranges still pass the rule above, bring Nov–Dec coverage closer to 80%, and do not worsen the interval score.

In [0]:
data["peak_scale"] = data["err_scale"] * (data["blend"] / data["level"]).clip(0.5, 3)
tested["peak_scale"] = data.loc[tested.index, "peak_scale"]


def add_ranges(frame, scale_col):
    """P10/P90 = blend + multiples x scale; multiples from the other backtest folds (all four for the holdout)."""
    out = frame.copy()
    ratio = (out["Weekly_Sales"] - out["blend"]) / out[scale_col]
    for split in TEST_SPLITS:
        low, high = ratio[out["split"].isin(FOLDS) & (out["split"] != split)].quantile([0.1, 0.9])
        rows = out["split"] == split
        out.loc[rows, "p10"] = out.loc[rows, "blend"] + low * out.loc[rows, scale_col]
        out.loc[rows, "p90"] = out.loc[rows, "blend"] + high * out.loc[rows, scale_col]
    out["inside"] = out["Weekly_Sales"].between(out["p10"], out["p90"])
    return out


peak = add_ranges(tested, "peak_scale")
versions = {"current": tested, "peak_aware": peak}
comparison = pd.DataFrame({name: interval_summary(f[f["period"] == "backtest"]) for name, f in versions.items()})
comparison.loc["nov_dec_coverage_%"] = [100 * f.loc[season_of(f) == "Nov-Dec", "inside"].mean() for f in versions.values()]
display(comparison.round(2))

new, old = comparison["peak_aware"], comparison["current"]
adopt = (new["lowest_size_group_%"] >= 75 and new["highest_size_group_%"] <= 85
         and 75 <= new["sales_weighted_%"] <= 85
         and abs(new["nov_dec_coverage_%"] - 80) < abs(old["nov_dec_coverage_%"] - 80)
         and new["interval_score"] <= old["interval_score"])
if adopt:
    tested = peak
pd.Series({"adopt_peak_aware_ranges": adopt})

All three conditions hold, so the peak-aware ranges are used from here on, including the stocking buffers. Nov–Dec coverage is still slightly below 80%.

Stocking: Orders are placed weekly and arrive after 3 weeks, so each order must cover 4 weeks of demand. Stock is set to the 4-week forecast plus a buffer, calibrated on 4-week totals (summing weekly P90s would overstock, because weekly errors partly cancel out). Demand is net sales over the 4 weeks, like the forecast, floored at zero.

Comparing stocking at P50, P75 and P90, assuming a lost sale costs 3×, 5× or 10× as much as a dollar of overstock. Inventory theory says the cheapest level is the quantile stockout cost ÷ (stockout cost + overstock cost): about P75 at 3:1, P83 at 5:1 and P91 at 10:1. A fine grid of levels tests this.

In [0]:
LEAD_WEEKS, REVIEW_WEEKS = 3, 1
COVER = LEAD_WEEKS + REVIEW_WEEKS
STOCK_LEVELS = {"P50": 0.5, "P75": 0.75, "P90": 0.9}
scale_col = "peak_scale" if adopt else "err_scale"

# one row per store-dept and start week: totals over the next 4 weeks (net of returns, like the forecast)
ahead = data.groupby(KEYS, observed=True)
windows = pd.DataFrame({
    "split": data["split"],
    "Date": data["Date"],
    "demand": sum(ahead["Weekly_Sales"].shift(-i) for i in range(COVER)).clip(lower=0),
    "forecast": sum(ahead["blend"].shift(-i) for i in range(COVER)),
    "scale": sum(ahead[scale_col].shift(-i) for i in range(COVER)),
    "one_split": ahead["split"].shift(-(COVER - 1)) == data["split"],
}).dropna()
windows = windows[windows["one_split"] & (windows["split"] != "train_only")].copy()
windows["ratio"] = (windows["demand"] - windows["forecast"]) / windows["scale"]


def stock_at(frame, q):
    """Order-up-to level at quantile q; each test period calibrated on the other backtest folds."""
    stock = pd.Series(np.nan, index=frame.index)
    for split in TEST_SPLITS:
        calibration = frame[frame["split"].isin(FOLDS) & (frame["split"] != split)]
        rows = frame["split"] == split
        stock[rows] = frame.loc[rows, "forecast"] + calibration["ratio"].quantile(q) * frame.loc[rows, "scale"]
    return stock.clip(lower=0)


for name, q in STOCK_LEVELS.items():
    windows[name] = stock_at(windows, q)

windows.groupby("split").size().rename("4-week windows")

In [0]:
COST_RATIOS = [3, 5, 10]


def stocking_costs(frame):
    """Per stock level, as % of 4-week demand: share of windows covered, fill rate, excess stock, total cost."""
    demand = frame["demand"].sum()
    rows = {}
    for name in STOCK_LEVELS:
        excess = (frame[name] - frame["demand"]).clip(lower=0).sum() / demand
        short = (frame["demand"] - frame[name]).clip(lower=0).sum() / demand
        rows[name] = {"windows_covered": (frame["demand"] <= frame[name]).mean(),
                      "fill_rate": 1 - short, "excess_stock": excess,
                      **{f"cost_{r}:1": excess + r * short for r in COST_RATIOS}}
    return 100 * pd.DataFrame(rows).T


backtest_windows = windows[windows["split"].isin(FOLDS)]
stocking_costs(backtest_windows).round(1)

In [0]:
# Fine grid of stock levels: which quantile is cheapest at each cost ratio?
grid = np.round(np.arange(0.50, 0.96, 0.01), 2)
demand = backtest_windows["demand"].sum()
grid_costs = {}
for q in grid:
    stock = stock_at(windows, q)[backtest_windows.index]
    excess = (stock - backtest_windows["demand"]).clip(lower=0).sum() / demand
    short = (backtest_windows["demand"] - stock).clip(lower=0).sum() / demand
    grid_costs[q] = {r: 100 * (excess + r * short) for r in COST_RATIOS}
grid_costs = pd.DataFrame(grid_costs).T

pd.DataFrame({
    "cheapest_quantile": grid_costs.idxmin(),
    "theory_quantile": [r / (r + 1) for r in COST_RATIOS],
    "cost_at_cheapest_%": grid_costs.min(),
    "cost_at_P75_%": grid_costs.loc[0.75],
    "cost_at_P90_%": grid_costs.loc[0.9],
}, index=COST_RATIOS).rename_axis("stockout : overstock cost").round(2)

In [0]:
cost_cols = [f"cost_{r}:1" for r in COST_RATIOS]
seasons = season_of(backtest_windows)

cheapest = backtest_windows.groupby(seasons).apply(lambda w: stocking_costs(w)[cost_cols].idxmin())
cheapest.loc["All backtest"] = stocking_costs(backtest_windows)[cost_cols].idxmin()
display(cheapest.rename_axis("cheapest stock level"))

# share of 4-week windows covered by season (targets: 75% for P75, 90% for P90)
backtest_windows.groupby(seasons).apply(
    lambda w: pd.Series({level: 100 * (w["demand"] <= w[level]).mean() for level in ["P75", "P90"]})).round(1)

**Recommendation: stock up to P75** of the 4-week forecast, assuming a lost sale costs about 3× as much as overstock.

- P75 is the cheapest of the three levels at 3:1 and 5:1, and in 3 of 4 seasons at 3:1. Compared with stocking to the forecast (P50), it raises the fill rate for a small amount of extra stock.
- If a stockout costs about 10× overstock (e.g. high-margin or promoted items), stock up to P90.
- On the fine grid, the cheapest level sits a little below theory (about P70, P79 and P88 instead of P75, P83 and P91), but the cost difference to P75 or P90 is small.

Caveats:
- The cost ratio is an assumption; the tables show the answer for each ratio.
- Each 4-week order is treated separately, ignoring leftover stock from earlier orders.
- Sales are in dollars, not units.
- P75 is not equally reliable across seasons: it covers about 69% of 4-week windows in Aug–Oct and 84% in May–Jul, against a target of 75%.

## Final Holdout

The last 12 weeks (Aug–Oct 2012), scored once with every decision fixed.

In [0]:
holdout = data[data["observed"] & (data["split"] == "holdout")]
display(pd.DataFrame({
    "WAPE %": {name: 100 * wape(holdout, col) for name, col in MODELS.items()},
}).round(1))

holdout_ranges = tested[tested["period"] == "holdout"]
display(interval_summary(holdout_ranges).round(2).rename("holdout ranges"))

stocking_costs(windows[windows["split"] == "holdout"]).round(1)

- The blend lowers WAPE from 9.5% (benchmark) to 8.3%, in line with the backtest. LightGBM alone is about the same (8.2%).
- The ranges pass the rule on the holdout too.
- P75 is again the cheapest level at 3:1 and 5:1, and P90 at 10:1.
- No decision in this notebook uses the holdout, but it was looked at during earlier work on the project, so it is not fully clean.

## Conclusion

**Can we forecast weekly sales per store and department 4 weeks ahead well enough to support inventory planning, especially in peak weeks?**

Yes for most of the year. In peak weeks the gain is smaller and less certain.

- **Accuracy:** the final forecast, a 50/50 blend of LightGBM and the adjusted seasonal naive with a Christmas shift, lowers backtest WAPE from 10.2% to 9.0% against the seasonal naive with the same shift, and from 9.5% to 8.3% on the holdout. The gain holds in every test period and season (95% interval 0.9–1.5 points). The blend is not significantly better than LightGBM alone.
- **Peak weeks:** on flagged holidays the blend is about level with the shifted seasonal naive (10.0% vs 10.2%), and it is better across peak season (10.4% vs 11.4%). LightGBM alone is weaker than the benchmark on holidays.
- **Uncertainty:** the forecast ranges pass the coverage rule on the backtest and the holdout. In Nov–Dec they are still too narrow, covering about 77% of weeks against 80%.
- **Decision:** stocking up to P75 of the 4-week forecast is the cheapest option when a lost sale costs about 3× as much as overstock. If it costs about 10×, stock up to P90.

**Limitations:**
- The backtest contains only one Thanksgiving and one Christmas, so holiday results are indicative.
- Easter is not aligned in the seasonal naive, and the 2-day Christmas shift for 2012 could not be tested.
- LightGBM under-forecasts, most for small series. The buffers absorb this, but it is not corrected.
- The peak adjustment to the ranges was added after the first range results. P75 coverage of 4-week demand varies by season (about 69% to 84%).
- Results cover forecasts up to 4 weeks ahead only. Costs are in dollars, not units, each order is treated separately, and the stockout-to-overstock cost ratio is an assumption.
- The model settings and the 0.5 blend weight were fixed, not tuned. The holdout was looked at during earlier work on the project.

## Model Tracking and Deployment

The forecast runs as a batch job: train on every observed week, forecast the next 4 weeks for every store-dept, check the result, and save it for the API. MLflow keeps two kinds of runs apart:

- Evaluation run: how the chosen setup performed in testing (backtest and holdout metrics and tables).
- Forecast run: the production model and the forecasts it produced. Every forecast row carries this run's ID, the model version and the time it was created.

On Databricks, runs go to an experiment in your workspace, the model to the Unity Catalog registry, and the forecasts to the `output` volume.

In [0]:
spark.sql("CREATE VOLUME IF NOT EXISTS workspace.forecasting.output")
user = spark.sql("SELECT current_user()").first()[0]

mlflow.set_registry_uri("databricks-uc")  # register models in Unity Catalog
mlflow.set_experiment(f"/Users/{user}/walmart-forecast")

OUTPUT = "/Volumes/workspace/forecasting/output/"
MODEL_NAME = "workspace.forecasting.walmart_lgbm"

pd.Series({"tracking": mlflow.get_tracking_uri(), "output folder": OUTPUT, "model name": MODEL_NAME})

In [0]:
# Production model: trained on every observed week, then used for the 4 weeks after the data ends
production_model = fit_lgbm(observed)

future_weeks = pd.date_range(weeks[-1], periods=HORIZON + 1, freq="7D")[1:]
holiday_flag = features.drop_duplicates("Date").set_index("Date")["IsHoliday"]
future_rows = (series.merge(pd.DataFrame({"Date": future_weeks}), how="cross")
               .merge(stores, on="Store", how="left")
               .assign(observed=False, gap=False, IsHoliday=lambda d: d["Date"].map(holiday_flag).astype(bool)))

scoring = build_features(pd.concat([df, future_rows], ignore_index=True).sort_values(KEYS + ["Date"], ignore_index=True))
for col in ["Store", "Dept", "Type"]:
    scoring[col] = scoring[col].astype("category")
scoring = add_baselines(scoring)

future = scoring["Date"].isin(future_weeks)
scoring.loc[future, "lgbm"] = production_model.predict(scoring.loc[future, FEATURES]) * scoring.loc[future, "level"]
scoring["blend"] = 0.5 * scoring["lgbm"] + 0.5 * scoring["asn_xmas"]
scoring = add_error_scale(scoring)
scoring["peak_scale"] = scoring["err_scale"] * (scoring["blend"] / scoring["level"]).clip(0.5, 3)

# weekly P10/P90 from the backtest errors (all four folds), as for the holdout
calibration = tested[tested["split"].isin(FOLDS)]
low, high = ((calibration["Weekly_Sales"] - calibration["blend"]) / calibration[scale_col]).quantile([0.1, 0.9])
forecasts = scoring.loc[future, ["Store", "Dept", "Date", "blend", scale_col]].rename(columns={"blend": "p50"})
forecasts["p10"] = forecasts["p50"] + low * forecasts[scale_col]
forecasts["p90"] = forecasts["p50"] + high * forecasts[scale_col]

# stock to hold for the whole 4-week window, from the backtest 4-week errors
stock_multiple = windows.loc[windows["split"].isin(FOLDS), "ratio"].quantile([0.75, 0.9])
window = forecasts.groupby(["Store", "Dept"], observed=True).agg(forecast=("p50", "sum"), scale=(scale_col, "sum"))
window["stock_p75"] = (window["forecast"] + stock_multiple[0.75] * window["scale"]).clip(lower=0)
window["stock_p90"] = (window["forecast"] + stock_multiple[0.9] * window["scale"]).clip(lower=0)

forecasts = (forecasts.merge(window[["stock_p75", "stock_p90"]].reset_index(), on=["Store", "Dept"])
             [["Store", "Dept", "Date", "p10", "p50", "p90", "stock_p75", "stock_p90"]]
             .astype({"Store": int, "Dept": int}))

weekly_total = forecasts.groupby("Date")[["p50"]].sum().div(1e6)
last_year = df.groupby("Date")["Weekly_Sales"].sum().div(1e6)
weekly_total["same_week_last_year"] = last_year.reindex(weekly_total.index - pd.Timedelta(weeks=52)).to_numpy()
display(weekly_total.round(1).rename(columns={"p50": "forecast"}).rename_axis("total sales ($M)"))
forecasts[(forecasts["Store"] == 1) & (forecasts["Dept"] == 1)].round(0)

In [0]:
# Evaluation run: how the chosen setup performed in testing
gain = block_bootstrap(weekly_errors(backtest, "asn_xmas", "blend"))

with mlflow.start_run(run_name="evaluation") as eval_run:
    mlflow.log_params({**{k: v for k, v in PARAMS.items() if k != "verbose"}, "n_rounds": N_ROUNDS,
                       "horizon_weeks": HORIZON, "blend_weight_lightgbm": 0.5, "peak_aware_ranges": adopt})
    mlflow.log_metrics({
        "backtest_wape_blend": 100 * wape(backtest, "blend"),
        "backtest_wape_benchmark": 100 * wape(backtest, "asn_xmas"),
        "backtest_gain_points": gain["gain_points"],
        "backtest_gain_ci_low": gain["ci_95_low"],
        "backtest_gain_ci_high": gain["ci_95_high"],
        "holdout_wape_blend": 100 * wape(holdout, "blend"),
        "holdout_wape_benchmark": 100 * wape(holdout, "asn_xmas"),
        "backtest_range_coverage_sales": interval_summary(tested[tested["period"] == "backtest"])["sales_weighted_%"],
        "holdout_range_coverage_sales": interval_summary(tested[tested["period"] == "holdout"])["sales_weighted_%"],
    })
    mlflow.log_text(score(data, MODELS).to_csv(), "results/backtest_scores.csv")
    mlflow.log_text(by_season.to_csv(), "results/backtest_by_season.csv")
    mlflow.log_text(stocking_costs(backtest_windows).to_csv(), "results/stocking_costs.csv")

pd.Series({"evaluation run_id": eval_run.info.run_id})

In [0]:
def check_forecasts(frame):
    """Stop before publishing if the forecasts look wrong."""
    weekly = frame.groupby("Date")["p50"].sum()
    last_year = df.groupby("Date")["Weekly_Sales"].sum().reindex(weekly.index - pd.Timedelta(weeks=52)).to_numpy()
    checks = pd.Series({
        "one row per store-dept and week": len(frame) == len(series) * HORIZON
                                           and not frame.duplicated(["Store", "Dept", "Date"]).any(),
        "no missing values": not frame.isna().any().any(),
        "p10 <= p50 <= p90": ((frame["p10"] <= frame["p50"]) & (frame["p50"] <= frame["p90"])).all(),
        "stock at P75 <= stock at P90": (frame["stock_p75"] <= frame["stock_p90"]).all(),
        "weekly totals within 25% of last year": (abs(weekly.to_numpy() / last_year - 1) < 0.25).all(),
    })
    if not checks.all():
        raise ValueError(f"Forecast checks failed: {list(checks[~checks].index)}")
    return checks


check_forecasts(forecasts).rename("passed")

In [0]:
class LevelScaledLGBM(mlflow.pyfunc.PythonModel):
    """The LightGBM forecast in dollars: sets the categorical columns, predicts sales / level, multiplies by level."""

    def __init__(self, booster, features, categories):
        self.booster, self.features, self.categories = booster, features, categories

    def predict(self, context, model_input, params=None):
        rows = model_input.copy()
        for col, values in self.categories.items():
            rows[col] = pd.Categorical(rows[col], categories=values)
        return self.booster.predict(rows[self.features]) * rows["level"].to_numpy()


categorical = ["Store", "Dept", "Type"]
model = LevelScaledLGBM(production_model, FEATURES, {c: observed[c].cat.categories for c in categorical})
example = scoring.loc[future, FEATURES + ["level"]].head(5).astype({"Store": int, "Dept": int, "Type": str})
signature = mlflow.models.infer_signature(example, model.predict(None, example))
forecast_file = OUTPUT + "forecasts.parquet"

# Forecast run: the production model and its forecasts, linked by run_id and model version
with mlflow.start_run(run_name="forecast") as forecast_run:
    mlflow.log_params({"first_week": str(future_weeks[0].date()), "last_week": str(future_weeks[-1].date()),
                       "evaluation_run_id": eval_run.info.run_id})
    model_info = mlflow.pyfunc.log_model("model", python_model=model, signature=signature, input_example=example,
                                         registered_model_name=MODEL_NAME, pip_requirements=["lightgbm==4.5.0", "pandas"])
    published = forecasts.assign(run_id=forecast_run.info.run_id,
                                 model_version=int(model_info.registered_model_version),
                                 created_at=pd.Timestamp.now(tz="UTC").floor("s"))
    published.to_parquet(forecast_file, index=False)
    mlflow.log_artifact(forecast_file, "forecasts")
    mlflow.log_metric("forecast_total_p50", published["p50"].sum())

published.head(4)

- The forecast covers Nov 2–23, 2012 for all 3,331 store-depts: P10, P50 and P90 per week, plus the stock to hold for the 4 weeks at P75 and P90.
- Total forecast sales follow last year's pattern, including the Thanksgiving week (Nov 23).
- The forecasts pass all checks before they are written. Each row records the forecast run, the model version and the creation time, so the file the API serves can be traced back to its run.
- The registered model is the LightGBM part only: it takes the 28 features plus the series level and returns sales in dollars. The blend, ranges and stock levels are computed in this notebook and saved in `forecasts.parquet`.